# Chapter 15 - Evaluation of Techniques

> We use numerous methodologies for the same examples, not simply for the
> purpose of demonstration, but because actuaries should use more than one
> method when analyzing unpaid claims. No single method can produce the best
> estimate in all situations.
>
> -- Friedland, Chapter 15

Chapter 15 does not introduce a new estimator. It brings the methods from
Chapters 7 through 14 together and asks whether they agree. This notebook
recreates the two comparison tables that can be built from work already in
the package:

- **U.S. Industry Auto** — IBNR and total unpaid from Development, Expected
  Claims, Bornhuetter-Ferguson, and Cape Cod, valued at 12/31/2007.
- **Changing conditions** — estimated IBNR across the U.S. PP Auto claim-ratio
  and case-outstanding scenarios and the U.S. Auto product-mix scenario,
  including Benktander.

The XYZ Insurer Exhibit I comparison, the Berquist-Sherman summaries, and the
DC Insurer monitoring exhibits depend on Chapters 11–13 and are left for a
later slice. Case Outstanding Development is omitted for the same reason.

Selections follow the earlier Friedland notebooks: Chapter 7 development
patterns (age-to-age factors rounded to three decimals), Chapter 8 expected
claim ratios, and the Chapter 9 device of folding a rounded percent unreported
back into an effective CDF so `BornhuetterFerguson` and `Benktander` match the
text.

In [1]:
import numpy as np
import pandas as pd
import chainladder as cl
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


def col(triangle):
    """Pull a 1-column triangle (latest diagonal, ultimate, IBNR) as a vector."""
    return triangle.to_frame(origin_as_datetime=False).iloc[:, 0].values


def as_apriori(triangle, values):
    """Broadcast a per-origin expected-claims vector onto a sample_weight triangle."""
    apriori = triangle.latest_diagonal.copy()
    apriori.iloc[0, 0] = np.asarray(values, dtype=float).reshape(apriori.shape)
    return apriori


def rounded_ldf_dev(triangle, n_periods, tail=1.0):
    """Chapter 7/10 selection: n-period simple average, constant tail, LDF rounded to 3dp."""
    dev = cl.TailConstant(tail=tail, projection_period=0).fit_transform(
        cl.Development(n_periods=n_periods, average="simple").fit_transform(triangle)
    )
    dev.ldf_ = dev.ldf_.round(3)
    return dev


def _simple_dev(triangle, n_periods, tail=1.0):
    return cl.TailConstant(tail=tail, projection_period=0).fit_transform(
        cl.Development(n_periods=n_periods, average="simple").fit_transform(triangle)
    )


def rounded_cdf_dev(triangle, n_periods, tail=1.0):
    """Apply a CDF rounded to three decimals via DevelopmentConstant."""
    dev = _simple_dev(triangle, n_periods, tail=tail)
    ages = [int(age) for age in triangle.development.values]
    cdf = np.maximum(
        dev.cdf_.to_frame(origin_as_datetime=False).values.flatten(), 1.0
    ).round(3)
    return cl.DevelopmentConstant(
        patterns=dict(zip(ages, cdf)), style="cdf"
    ).fit_transform(triangle)


def bf_style_dev(triangle, n_periods, tail=1.0):
    """Chapter 9: fold a rounded percent unreported / unpaid back into an effective CDF."""
    dev = _simple_dev(triangle, n_periods, tail=tail)
    ages = [int(age) for age in triangle.development.values]
    cdf = np.maximum(
        dev.cdf_.to_frame(origin_as_datetime=False).values.flatten(), 1.0
    ).round(3)
    pct = np.round(1 - 1 / cdf, 3)
    effective = 1.0 / (1.0 - pct)
    return cl.DevelopmentConstant(
        patterns=dict(zip(ages, effective)), style="cdf"
    ).fit_transform(triangle)


def ibnr_from_ultimate(ultimate, reported):
    """IBNR is ultimate minus reported, including for paid-basis methods."""
    return float(np.nansum(
        np.nan_to_num(col(ultimate)) - col(reported.latest_diagonal)
    ))


def unpaid_from_ultimate(ultimate, paid):
    return float(np.nansum(
        np.nan_to_num(col(ultimate)) - col(paid.latest_diagonal)
    ))

## U.S. Industry Auto

For the consolidated U.S. private passenger automobile portfolio the methods
agree, as the text expects given the volume of business. The table below is
the Chapter 15 summary of estimated unpaid claims as of 12/31/2007, in
billions of dollars.

Development, Expected Claims, and Bornhuetter-Ferguson reuse the Chapter 8
selected CDFs and the Chapter 8 selected claim ratios (75% for 1998–2002,
65% for 2003–2007). Cape Cod reuses the Chapter 7 three-year simple-average
reported pattern with a 1.000 tail, rounding the age-to-age factors to three
decimals before `CapeCod` derives the all-years claim ratio.

In [2]:
ia = cl.load_sample("friedland_us_industry_auto")
ia_reported = ia["Reported Claims"]
ia_paid = ia["Paid Claims"]
ia_premium = ia["Earned Premium"].latest_diagonal

# Chapter 8 selected CDFs (three-year simple average, rounded, with tails).
ia_reported_pattern = {
    12: 1.292, 24: 1.110, 36: 1.051, 48: 1.023, 60: 1.011,
    72: 1.006, 84: 1.003, 96: 1.001, 108: 1.000, 120: 1.000,
}
ia_paid_pattern = {
    12: 2.390, 24: 1.404, 36: 1.184, 48: 1.085, 60: 1.040,
    72: 1.020, 84: 1.011, 96: 1.006, 108: 1.004, 120: 1.002,
}
ia_rep_dev = cl.DevelopmentConstant(
    patterns=ia_reported_pattern, style="cdf"
).fit_transform(ia_reported)
ia_paid_dev = cl.DevelopmentConstant(
    patterns=ia_paid_pattern, style="cdf"
).fit_transform(ia_paid)

ia_cl_reported = cl.Chainladder().fit(ia_rep_dev)
ia_cl_paid = cl.Chainladder().fit(ia_paid_dev)

# Chapter 8 selected claim ratios applied to earned premium.
ia_claim_ratio = np.array(
    [0.75, 0.75, 0.75, 0.75, 0.75, 0.65, 0.65, 0.65, 0.65, 0.65]
)
ia_el = cl.ExpectedLoss(apriori=1).fit(
    ia_reported,
    sample_weight=ia_premium * ia_claim_ratio.reshape(1, 1, -1, 1),
)

# Chapter 8 expected claims carried into the Chapter 9 BF projection.
ia_expected = np.array(
    [51430657, 51408736, 51680983, 54408716, 59421665,
     56318302, 59646290, 61174953, 61926981, 61864556],
    dtype=float,
)
ia_apriori = as_apriori(ia_reported, ia_expected)
ia_bf_reported = cl.BornhuetterFerguson(apriori=1.0).fit(
    ia_rep_dev, sample_weight=ia_apriori
)
ia_bf_paid = cl.BornhuetterFerguson(apriori=1.0).fit(
    ia_paid_dev, sample_weight=ia_apriori
)

ia_cc_dev = rounded_ldf_dev(ia_reported, n_periods=3, tail=1.000)
ia_cc = cl.CapeCod().fit(ia_cc_dev, sample_weight=ia_premium)

ia_methods = {
    "Development – Reported": ia_cl_reported.ultimate_,
    "Development – Paid": ia_cl_paid.ultimate_,
    "Expected Claims": ia_el.ultimate_,
    "Bornhuetter-Ferguson – Reported": ia_bf_reported.ultimate_,
    "Bornhuetter-Ferguson – Paid": ia_bf_paid.ultimate_,
    "Cape Cod": ia_cc.ultimate_,
}

ia_raw = pd.DataFrame({
    method: {
        "IBNR": ibnr_from_ultimate(ultimate, ia_reported),
        "Total Unpaid": unpaid_from_ultimate(ultimate, ia_paid),
    }
    for method, ultimate in ia_methods.items()
}).T

# Friedland prints this table in $ billions (the sample is in $000).
ia_billions = (ia_raw / 1e6).round(0).astype(int)
ia_billions.columns = ["IBNR", "Total"]
display(ia_billions)

,IBNR,Total
Development – Reported,26,71
Development – Paid,29,74
Expected Claims,26,71
Bornhuetter-Ferguson – Reported,26,71
Bornhuetter-Ferguson – Paid,27,73
Cape Cod,27,73


### Reconciliation to Friedland

The printed Chapter 15 Industry Auto table is in billions of dollars. After
converting the $000 sample totals, every method rounds to the published IBNR
and total unpaid. Case Outstanding Development (printed 24 / 70) is omitted
until Chapter 12 is available.

In [3]:
ia_printed = pd.DataFrame(
    {
        "IBNR": [26, 29, 26, 26, 27, 27],
        "Total": [71, 74, 71, 71, 73, 73],
    },
    index=ia_billions.index,
)
assert (ia_billions - ia_printed).abs().max().max() <= 1

## Changing Conditions

Chapters 7 through 10 run the same methods through four U.S. PP Auto
environments and a combined private-passenger / commercial auto portfolio.
When the portfolio is in a steady state every method recovers the true IBNR.
When claim ratios, case outstanding strength, or product mix change, the
methods diverge. Chapter 15 summarises that divergence.

The first line of the table is the true IBNR required in each scenario (from
the Chapter 8 exhibits). The remaining rows are the IBNR implied by each
technique: ultimate minus reported, including for paid-basis methods.

U.S. PP Auto uses a 70% expected claim ratio and a five-year simple-average
development selection. The product-mix scenario uses a 75% expected claim
ratio on the same five-year selection. Benktander is the two-iteration form
(`n_iters=2`), which sits between Bornhuetter-Ferguson and chain ladder.

In [4]:
def scenario_ibnr(triangle, claim_ratio, n_periods=5):
    """IBNR for each Chapter 15 technique on one changing-conditions portfolio."""
    reported = triangle["Reported Claims"]
    paid = triangle["Paid Claims"]
    premium = triangle["Earned Premium"].latest_diagonal
    expected = np.round(claim_ratio * col(premium))
    apriori = as_apriori(reported, expected)

    cl_reported = cl.Chainladder().fit(rounded_cdf_dev(reported, n_periods))
    cl_paid = cl.Chainladder().fit(rounded_cdf_dev(paid, n_periods))
    el = cl.ExpectedLoss(apriori=claim_ratio).fit(
        reported, sample_weight=np.round(premium, 0)
    )
    bf_reported = cl.BornhuetterFerguson(apriori=1.0).fit(
        bf_style_dev(reported, n_periods), sample_weight=apriori
    )
    bf_paid = cl.BornhuetterFerguson(apriori=1.0).fit(
        bf_style_dev(paid, n_periods), sample_weight=apriori
    )
    bk_reported = cl.Benktander(apriori=1.0, n_iters=2).fit(
        bf_style_dev(reported, n_periods), sample_weight=apriori
    )
    bk_paid = cl.Benktander(apriori=1.0, n_iters=2).fit(
        bf_style_dev(paid, n_periods), sample_weight=apriori
    )
    cc = cl.CapeCod().fit(
        rounded_ldf_dev(reported, n_periods), sample_weight=premium
    )
    return {
        "Development – Reported": ibnr_from_ultimate(cl_reported.ultimate_, reported),
        "Development – Paid": ibnr_from_ultimate(cl_paid.ultimate_, reported),
        "Expected Claims": ibnr_from_ultimate(el.ultimate_, reported),
        "Bornhuetter-Ferguson – Reported": ibnr_from_ultimate(
            bf_reported.ultimate_, reported
        ),
        "Bornhuetter-Ferguson – Paid": ibnr_from_ultimate(bf_paid.ultimate_, reported),
        "Benktander – Reported": ibnr_from_ultimate(bk_reported.ultimate_, reported),
        "Benktander – Paid": ibnr_from_ultimate(bk_paid.ultimate_, reported),
        "Cape Cod": ibnr_from_ultimate(cc.ultimate_, reported),
    }


pp_scenarios = {
    "Increasing Claim Ratios": (
        cl.load_sample("friedland_uspp_auto_increasing_claim"), 0.70, 601984,
    ),
    "Increasing Case Outstanding Strength": (
        cl.load_sample("friedland_uspp_auto_increasing_case"), 0.70, 253336,
    ),
    "Increasing Claim Ratios and Case Outstanding Strength": (
        cl.load_sample("friedland_uspp_increasing_claim_case"), 0.70, 347660,
    ),
}
us_auto = cl.load_sample("friedland_us_auto")

cc_raw = {}
for name, (triangle, claim_ratio, true_ibnr) in pp_scenarios.items():
    row = scenario_ibnr(triangle, claim_ratio)
    row["True IBNR"] = true_ibnr
    cc_raw[name] = row

mix_row = scenario_ibnr(us_auto.loc["Changing Product Mix"], 0.75)
mix_row["True IBNR"] = 2391084
cc_raw["Changing Product Mix"] = mix_row

method_order = [
    "True IBNR",
    "Development – Reported",
    "Development – Paid",
    "Expected Claims",
    "Bornhuetter-Ferguson – Reported",
    "Bornhuetter-Ferguson – Paid",
    "Benktander – Reported",
    "Benktander – Paid",
    "Cape Cod",
]
# Friedland prints this table in thousands of the $000 sample (nearest unit).
cc_table = (
    pd.DataFrame(cc_raw).T[method_order].T / 1000
).round(0).astype(int)
display(cc_table)

,Increasing Claim Ratios,Increasing Case Outstanding Strength,Increasing Claim Ratios and Case Outstanding Strength,Changing Product Mix
True IBNR,602,253,348,2391
Development – Reported,602,499,687,2146
Development – Paid,601,252,347,1702
Expected Claims,-843,253,-1097,2167
Bornhuetter-Ferguson – Reported,439,458,458,2165
Bornhuetter-Ferguson – Paid,159,253,-96,1980
Benktander – Reported,573,491,644,2154
Benktander – Paid,406,253,151,1876
Cape Cod,507,465,538,2166


### Reconciliation to Friedland

The Increasing Claim Ratios column uses the same clean sample as Chapters 8–10
and reconciles to the printed table. The two case-outstanding columns and the
Changing Product Mix column are checked with a relative tolerance. Chapters 9
and 10 already note that `friedland_uspp_auto_increasing_case`,
`friedland_uspp_increasing_claim_case`, and the combined-auto sample differ
slightly from the text; the same gap appears here.

In [5]:
cc_printed = pd.DataFrame(
    {
        "Increasing Claim Ratios": [602, 602, 602, -843, 439, 159, 573, 406, 506],
        "Increasing Case Outstanding Strength": [253, 501, 253, 253, 458, 253, 492, 253, 470],
        "Increasing Claim Ratios and Case Outstanding Strength": [
            348, 694, 348, -1097, 460, -96, 648, 151, 546,
        ],
        "Changing Product Mix": [2391, 2153, 1723, 2167, 2168, 1991, 2159, 1893, 2168],
    },
    index=method_order,
)

# Clean column: every method rounds to the printed IBNR.
assert (
    cc_table["Increasing Claim Ratios"] - cc_printed["Increasing Claim Ratios"]
).abs().max() <= 2

# Remaining columns: Chapters 9 and 10 already note that the case-outstanding
# and product-mix samples differ slightly from the text.
other = [c for c in cc_printed.columns if c != "Increasing Claim Ratios"]
assert np.allclose(
    cc_table[other].astype(float), cc_printed[other].astype(float), rtol=0.03, atol=10
)